In [ ]:
'''
Лабораторная работа: Анализ табличных данных и временных рядов (Pandas)

Цель работы
Освоить инструменты библиотеки Pandas для обработки, очистки, агрегации и визуализации табличных медицинских данных. Научиться работать с временными рядами, оконными функциями, объединением таблиц и базовым разведочным анализом (EDA).

Постановка задачи
В рамках клинического исследования эффективности нового препарата кардиологического профиля пациенты носят смарт-браслеты. Устройства каждую минуту фиксируют частоту сердечных сокращений (ЧСС) и уровень кислорода в крови (SpO2).

Датчики периодически теряют контакт с кожей, из-за чего в данных появляются пропуски (NaN). Кроме того, сырые данные содержат аппаратный шум.
Необходимо написать скрипт подготовки и анализа этих данных.

Этапы работы:

Генерация и I/O (ввод/вывод).
Создать датафрейм с временным рядом для нескольких пациентов. 
Искусственно внести пропуски в данные. 
Сгенерировать вторую таблицу — справочник пациентов (возраст, диагноз). 
Сохранить основную таблицу в CSV и сразу же считать ее обратно 
(имитация реального процесса загрузки сырых логов).

Очистка данных и объединение (Data Cleaning & Merge).
Преобразовать строковые даты в формат datetime и сделать их индексом датафрейма. 
Соединить таблицу логов со справочником пациентов по patient_id (аналог SQL JOIN). 
Заполнить пропуски в показаниях датчиков методом линейной интерполяции.

Оконные функции и фильтрация (Слайсинг).
Сырой сигнал ЧСС зашумлен. Необходимо создать новую колонку со сглаженным значением ЧСС, 
используя скользящее среднее (rolling window) за 15 минут.
С помощью логического индексирования (булевых масок) найти все периоды тахикардии 
(сглаженный ЧСС > 100 уд/мин).

Группировка и агрегация (Groupby).
Сгруппировать данные по пациентам и дням. Вычислить для каждого пациента его среднесуточный пульс, 
минимальный SpO2 и суммарное время в состоянии тахикардии.

Визуализация средствами Pandas.
Построить дашборд из трех графиков, используя встроенный метод .plot() датафрейма:

Линейный график сырого и сглаженного пульса для одного выбранного пациента за конкретный день.
Столбчатая диаграмма (Bar chart) минимального SpO2 по пациентам.
Гистограмма распределения возраста пациентов в исследовании.
Требования:
Отказ от ручного перебора строк (никаких iterrows, если это не обосновано). 
Максимальное использование встроенных векторизованных методов Pandas.
'''

In [ ]:
"""
Лабораторная работа: Pandas
Тема: Анализ данных носимых медицинских устройств (Time Series EDA)

Ваша задача: заполнить все пропуски (TODO), используя векторизованные 
методы и функции библиотеки Pandas. Использование циклов for для 
обработки строк датафрейма строго запрещено.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==================== 1. ГЕНЕРАЦИЯ ДАННЫХ И I/O ====================

def generate_and_save_data(filename="wearable_log.csv"):
    """Генерация логов браслета и сохранение в файл."""
    np.random.seed(42)
    dates = pd.date_range(start="2023-10-01", periods=3*24*60, freq="1min")
    
    data = []
    for patient_id in ["P001", "P002", "P003"]:
        base_hr = np.random.randint(65, 85)
        hr_signal = base_hr + np.cumsum(np.random.normal(0, 0.5, len(dates))) + np.random.normal(0, 2, len(dates))
        spo2_signal = 98 - np.abs(np.random.normal(0, 2, len(dates)))
        
        df_temp = pd.DataFrame({
            "timestamp": dates,
            "patient_id": patient_id,
            "hr_raw": hr_signal,
            "spo2": spo2_signal
        })
        data.append(df_temp)
        
    # TODO: Объедините список датафреймов data в один большой датафрейм df_logs.
    # Используйте функцию pd.concat, игнорируя старые индексы (ignore_index=True).
    df_logs = ...
    
    mask = np.random.rand(len(df_logs)) < 0.05
    df_logs.loc[mask, ["hr_raw", "spo2"]] = np.nan
    
    # TODO: Сохраните датафрейм df_logs в CSV файл с именем filename. 
    # Отключите сохранение индексов (index=False).
    pass

def create_patient_metadata() -> pd.DataFrame:
    """Создание справочника пациентов."""
    data = {
        "patient_id": ["P001", "P002", "P003"],
        "age": [45, 62, 38],
        "diagnosis": ["Healthy", "Arrhythmia", "Healthy"],
        "medication": ["None", "Beta-blockers", "None"]
    }
    # TODO: Создайте и верните pandas DataFrame на основе словаря data.
    return ...

# ==================== 2. ОЧИСТКА И ОБЪЕДИНЕНИЕ ====================

def load_and_preprocess(filename: str, df_meta: pd.DataFrame) -> pd.DataFrame:
    """Чтение логов, джоин справочника и очистка от пропусков."""
    
    # TODO: Считайте CSV файл filename в датафрейм df
    df = ...
    
    # TODO: Преобразуйте колонку 'timestamp' из строкового формата в формат datetime.
    # Используйте pd.to_datetime()
    df['timestamp'] = ...
    
    # TODO: Выполните объединение (merge) таблицы df со справочником df_meta.
    # Ключ объединения — колонка 'patient_id'. Тип объединения — LEFT JOIN.
    df = ...
    
    # TODO: Отсортируйте данные сначала по 'patient_id', затем по 'timestamp'.
    df = ...
    
    # TODO: Сделайте колонку 'timestamp' индексом датафрейма (set_index).
    # Это необходимо для удобной работы с временными рядами.
    df = ...
    
    # TODO: Заполните пропуски (NaN) в колонках 'hr_raw' и 'spo2'.
    # Так как это временной ряд, примените метод линейной интерполяции (.interpolate()).
    # Применяйте метод только к указанным числовым колонкам.
    df[['hr_raw', 'spo2']] = ...
    
    return df

# ==================== 3. ОКОННЫЕ ФУНКЦИИ И СЛАЙСИНГ ====================

def analyze_signals(df: pd.DataFrame) -> pd.DataFrame:
    """Сглаживание сигнала и поиск аномалий."""
    
    # TODO: Рассчитайте скользящее среднее (rolling average) пульса за 15 минут.
    # ВАЖНО: нужно сгруппировать данные по 'patient_id', взять колонку 'hr_raw',
    # применить окно размером 15 (.rolling(15)) и вычислить среднее (.mean()).
    df['hr_smooth'] = ...
    
    # TODO: После скользящего среднего первые 14 значений каждого пациента стали NaN.
    # Заполните их методом обратного заполнения (.bfill()).
    df['hr_smooth'] = ...
    
    # TODO: Создайте новую колонку 'is_tachycardia' типа bool.
    # Она должна содержать True, если сглаженный пульс ('hr_smooth') строго больше 100.
    df['is_tachycardia'] = ...
    
    return df

# ==================== 4. АГРЕГАЦИЯ ====================

def calculate_daily_stats(df: pd.DataFrame) -> pd.DataFrame:
    """Агрегация суточной статистики по каждому пациенту."""
    
    # TODO: Создайте временную колонку 'date', извлекая только дату из индекса.
    # Подсказка: используйте df.index.date
    df['date'] = ...
    
    # TODO: Сгруппируйте датафрейм по двум колонкам: 'patient_id' и 'date'.
    # Для сгруппированных данных вызовите метод .agg() и рассчитайте:
    # 1. Для 'hr_smooth' -> среднее ('mean')
    # 2. Для 'spo2' -> минимум ('min')
    # 3. Для 'is_tachycardia' -> сумму ('sum') (это даст кол-во минут тахикардии в день)
    daily_stats = ...
    
    # Удаляем временную колонку 'date' из исходного df, чтобы не засорять его
    df.drop(columns=['date'], inplace=True)
    
    return daily_stats

# ==================== 5. ВИЗУАЛИЗАЦИЯ ====================

def plot_dashboard(df: pd.DataFrame, daily_stats: pd.DataFrame):
    """Визуализация данных встроенными средствами Pandas."""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle("Анализ данных клинического мониторинга", fontsize=16)
    
    # --- График 1: Сырой vs Сглаженный пульс (P002 за 2023-10-01) ---
    ax1 = axes[0, 0]
    
    # TODO: Отфильтруйте датафрейм, оставив только строки пациента 'P002'.
    # TODO: Используйте метод .loc[] для среза данных только за дату '2023-10-01'.
    # TODO: Выберите только две колонки: ['hr_raw', 'hr_smooth'].
    # TODO: Вызовите метод .plot(ax=ax1, alpha=0.7)
    
    # df_p002 = ...
    # df_p002_day1 = ...
    # df_p002_day1[...].plot(...)
    
    ax1.set_title("Пульс (P002): Сырой vs Сглаженный")
    ax1.set_ylabel("ЧСС (уд/мин)")
    
    # --- График 2: Минимальный SpO2 по пациентам ---
    ax2 = axes[0, 1]
    
    # TODO: Возьмите агрегированный датафрейм daily_stats.
    # TODO: Сгруппируйте его по 'patient_id' и найдите минимум по колонке 'spo2'.
    # TODO: Постройте столбчатую диаграмму, вызвав .plot.bar(ax=ax2, color=['green', 'orange', 'blue'])
    pass # ЗАМЕНИТЬ НА ВАШ КОД
    
    ax2.set_title("Абсолютный минимум SpO2 за весь период")
    ax2.set_ylabel("SpO2 (%)")
    ax2.set_ylim(85, 100)
    
    # --- График 3: Распределение возраста пациентов ---
    ax3 = axes[1, 0]
    
    # TODO: Получите датафрейм только с уникальными пациентами и их возрастом.
    # Для этого: выделите колонки ['patient_id', 'age'] и примените .drop_duplicates().
    # TODO: Выберите колонку 'age' и постройте гистограмму: .plot.hist(ax=ax3, bins=5, color='purple')
    pass # ЗАМЕНИТЬ НА ВАШ КОД
    
    ax3.set_title("Распределение возраста участников")
    ax3.set_xlabel("Возраст (лет)")
    
    # --- График 4: Свободный (пустой) ---
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    # TODO: Отобразите графики
    pass

# ==================== ГЛАВНЫЙ ЦИКЛ ====================

def main():
    filename = "wearable_log.csv"
    
    print("1. Генерация и I/O...")
    generate_and_save_data(filename)
    df_meta = create_patient_metadata()
    
    print("2. Загрузка и предобработка...")
    df = load_and_preprocess(filename, df_meta)
    print(f"Размерность данных: {df.shape}")
    
    print("3. Расчет скользящего среднего и поиск аномалий...")
    df = analyze_signals(df)
    
    print("4. Суточная агрегация...")
    daily_stats = calculate_daily_stats(df)
    print("\nФрагмент агрегированной статистики:")
    print(daily_stats.head())
    
    print("\n5. Отрисовка дашборда...")
    plot_dashboard(df, daily_stats)

if __name__ == "__main__":
    main()
